# CS-781 Phase 1 — Architecture 2 (encoder–decoder): tokenizer and data preparation

In this notebook I build everything my encoder–decoder needs before training (training itself is in `archit-2-v2`):
1. Environment setup and reproducibility (seed 42), GPU check.
2. A ~100 MB tokenizer corpus sampled from the four official Phase-1 sources.
3. A 32k byte-level BPE tokenizer trained on that corpus (special tokens `<pad>`, `<s>`, `</s>`, `<unk>`), with a round-trip test.
4. Encoder–decoder training pairs: 512-word prompt → next 128 words, with token-length analysis to set the encoder and decoder lengths.
5. Tokenised data shards saved for training.


In [1]:
# ============================================================
# STEP 1: BASIC ENVIRONMENT SETUP
# Import core libraries, set reproducibility, and check GPUs
# ============================================================

import os
import random
import numpy as np
import torch


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Environment check
# ------------------------------------------------------------

print("=" * 70)
print("ARCHITECTURE 2: ENCODER-DECODER TRANSFORMER")
print("=" * 70)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(i)

        print(f"\nGPU {i}")
        print(f"Name            : {properties.name}")
        print(f"Memory          : {properties.total_memory / (1024**3):.2f} GB")

print("\nEnvironment setup complete.")

ARCHITECTURE 2: ENCODER-DECODER TRANSFORMER
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU count       : 2

GPU 0
Name            : Tesla T4
Memory          : 14.56 GB

GPU 1
Name            : Tesla T4
Memory          : 14.56 GB

Environment setup complete.


In [2]:
# ============================================================
# STEP 2: HUGGING FACE AUTHENTICATION
# Read the HF token securely from Kaggle Secrets and log in
# ============================================================

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login


# ------------------------------------------------------------
# Read Hugging Face token from Kaggle Secrets
# ------------------------------------------------------------

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF")

if not hf_token:
    raise ValueError("Hugging Face token was not found in Kaggle Secrets.")


# ------------------------------------------------------------
# Authenticate with Hugging Face
# ------------------------------------------------------------

login(
    token=hf_token,
    add_to_git_credential=False
)

print("=" * 70)
print("HUGGING FACE AUTHENTICATION")
print("=" * 70)
print("Authentication successful.")

HUGGING FACE AUTHENTICATION
Authentication successful.


In [3]:
# ============================================================
# STEP 3: DEFINE PHASE-1 DATA SOURCES
# Set the official Hugging Face paths for all four legal sources
# ============================================================

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_BASE_URL = (
    "https://huggingface.co/datasets/"
    f"{HF_DATASET_REPO}/resolve/main/"
    "Phase_1_Pretraining/Train/"
)


# ------------------------------------------------------------
# Four official Phase-1 training sources
# ------------------------------------------------------------

TRAIN_SOURCES = {
    "acts": HF_BASE_URL + "train_acts.json",
    "high_courts": HF_BASE_URL + "train_high_courts.json",
    "supreme_court": HF_BASE_URL + "train_supreme_court.json",
    "tribunals": HF_BASE_URL + "train_tribunals.json",
}


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("PHASE-1 TRAINING SOURCES")
print("=" * 70)

for source_name, source_url in TRAIN_SOURCES.items():
    print(f"{source_name:<15} -> {source_url}")

print(f"\nTotal sources: {len(TRAIN_SOURCES)}")

PHASE-1 TRAINING SOURCES
acts            -> https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json
high_courts     -> https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_high_courts.json
supreme_court   -> https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_supreme_court.json
tribunals       -> https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_tribunals.json

Total sources: 4


In [4]:
# ============================================================
# STEP 4A: INSTALL STREAMING JSON LIBRARY
# Install ijson for memory-efficient JSON streaming
# ============================================================

!pip install -q ijson

print("ijson installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 2.7 MB/s eta 0:00:00
ijson installed successfully.


In [5]:
# ============================================================
# STEP 4: STREAM JSON DOCUMENTS
# Read large Hugging Face JSON files without downloading them
# ============================================================

import requests
import ijson


def stream_json_documents(url, hf_token):

    # Authentication header
    headers = {
        "Authorization": f"Bearer {hf_token}"
    }

    # Open remote file as a stream
    response = requests.get(
        url,
        headers=headers,
        stream=True
    )

    response.raise_for_status()

    # Read one JSON object at a time
    for document in ijson.items(response.raw, "item"):
        yield document


print("=" * 70)
print("STREAMING READER")
print("=" * 70)
print("Streaming JSON reader created successfully.")

STREAMING READER
Streaming JSON reader created successfully.


In [6]:
# ============================================================
# STEP 5: INSPECT ONE DOCUMENT FROM EACH SOURCE
# Check the structure and text fields before corpus creation
# ============================================================

print("=" * 70)
print("SOURCE DOCUMENT INSPECTION")
print("=" * 70)


for source_name, source_url in TRAIN_SOURCES.items():

    print("\n" + "-" * 70)
    print(f"SOURCE: {source_name}")
    print("-" * 70)

    # Read only the first document
    document = next(
        stream_json_documents(
            source_url,
            hf_token
        )
    )

    # Show available fields
    print("Fields:", list(document.keys()))

    # Show short preview of each field
    for key, value in document.items():

        preview = str(value).replace("\n", " ")

        if len(preview) > 300:
            preview = preview[:300] + "..."

        print(f"\n{key}:")
        print(preview)

SOURCE DOCUMENT INSPECTION

----------------------------------------------------------------------
SOURCE: acts
----------------------------------------------------------------------
Fields: ['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars']

title:
Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996

category:
Greater Bengaluru City Corporation - Act

year:
1972

url:
https://indiankanoon.org/doc/179573747/

text:
Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu...

word_count:
1277

text_length_chars:
7784

----------------------------------------------------------------------
SOURCE: high_courts
----------------------------------------------------------------------
Fields: ['

In [7]:
# ============================================================
# STEP 6: TOKENIZER CORPUS CONFIGURATION
# Use balanced text samples from all four legal sources
# ============================================================

import os


# ------------------------------------------------------------
# Corpus size
# ------------------------------------------------------------

MB = 1024 * 1024

TARGET_BYTES_PER_SOURCE = 25 * MB

TOKENIZER_CORPUS_DIR = "/kaggle/working/tokenizer_data"
TOKENIZER_CORPUS_PATH = os.path.join(
    TOKENIZER_CORPUS_DIR,
    "tokenizer_corpus.txt"
)

os.makedirs(
    TOKENIZER_CORPUS_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("TOKENIZER CORPUS CONFIGURATION")
print("=" * 70)

print(f"Sources                 : {len(TRAIN_SOURCES)}")
print(f"Target per source       : {TARGET_BYTES_PER_SOURCE / MB:.0f} MB")
print(
    f"Approx total corpus     : "
    f"{TARGET_BYTES_PER_SOURCE * len(TRAIN_SOURCES) / MB:.0f} MB"
)
print(f"Output file             : {TOKENIZER_CORPUS_PATH}")

TOKENIZER CORPUS CONFIGURATION
Sources                 : 4
Target per source       : 25 MB
Approx total corpus     : 100 MB
Output file             : /kaggle/working/tokenizer_data/tokenizer_corpus.txt


In [8]:
# ============================================================
# STEP 7: BUILD TOKENIZER TRAINING CORPUS
# Stream 25 MB from each source and merge into one text file
# ============================================================

print("=" * 70)
print("BUILDING TOKENIZER CORPUS")
print("=" * 70)


source_stats = {}


with open(
    TOKENIZER_CORPUS_PATH,
    "w",
    encoding="utf-8"
) as output_file:

    for source_name, source_url in TRAIN_SOURCES.items():

        print(f"\nProcessing: {source_name}")

        collected_bytes = 0
        document_count = 0

        for document in stream_json_documents(
            source_url,
            hf_token
        ):

            text = document.get("text", "")

            if not isinstance(text, str):
                continue

            text = text.strip()

            if not text:
                continue

            # Write document to combined corpus
            output_file.write(text)
            output_file.write("\n\n")

            collected_bytes += len(
                text.encode("utf-8")
            )

            document_count += 1

            # Stop after reaching target
            if collected_bytes >= TARGET_BYTES_PER_SOURCE:
                break

        source_stats[source_name] = {
            "documents": document_count,
            "bytes": collected_bytes
        }

        print(
            f"Collected: "
            f"{collected_bytes / MB:.2f} MB "
            f"from {document_count:,} documents"
        )


print("\n" + "=" * 70)
print("TOKENIZER CORPUS COMPLETE")
print("=" * 70)

total_size = os.path.getsize(
    TOKENIZER_CORPUS_PATH
)

print(f"Final corpus size: {total_size / MB:.2f} MB")
print(f"Saved at         : {TOKENIZER_CORPUS_PATH}")

BUILDING TOKENIZER CORPUS

Processing: acts
Collected: 25.03 MB from 601 documents

Processing: high_courts
Collected: 25.00 MB from 10,028 documents

Processing: supreme_court
Collected: 25.14 MB from 757 documents

Processing: tribunals
Collected: 25.01 MB from 947 documents

TOKENIZER CORPUS COMPLETE
Final corpus size: 100.21 MB
Saved at         : /kaggle/working/tokenizer_data/tokenizer_corpus.txt


In [9]:
# ============================================================
# STEP 8: INITIALIZE BPE TOKENIZER
# ============================================================

from tokenizers import ByteLevelBPETokenizer


# ------------------------------------------------------------
# Initialize tokenizer
# ------------------------------------------------------------

tokenizer = ByteLevelBPETokenizer()


print("=" * 70)
print("BPE TOKENIZER INITIALIZATION")
print("=" * 70)

print("Tokenizer type : ByteLevelBPETokenizer")
print("Status         : Initialized successfully")
print(f"Training corpus: {TOKENIZER_CORPUS_PATH}")

BPE TOKENIZER INITIALIZATION
Tokenizer type : ByteLevelBPETokenizer
Status         : Initialized successfully
Training corpus: /kaggle/working/tokenizer_data/tokenizer_corpus.txt


In [10]:
# ============================================================
# STEP 9: TRAIN BPE TOKENIZER
# Learn vocabulary and merge rules from the legal corpus
# ============================================================

VOCAB_SIZE = 32_000
MIN_FREQUENCY = 2

SPECIAL_TOKENS = [
    "<pad>",
    "<s>",
    "</s>",
    "<unk>"
]


# ------------------------------------------------------------
# Train tokenizer
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING BPE TOKENIZER")
print("=" * 70)

tokenizer.train(
    files=[TOKENIZER_CORPUS_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS
)


# ------------------------------------------------------------
# Check trained vocabulary
# ------------------------------------------------------------

actual_vocab_size = tokenizer.get_vocab_size()

print("\nTraining complete.")
print(f"Requested vocabulary size : {VOCAB_SIZE:,}")
print(f"Actual vocabulary size    : {actual_vocab_size:,}")
print(f"Special tokens            : {SPECIAL_TOKENS}")

TRAINING BPE TOKENIZER




Training complete.
Requested vocabulary size : 32,000
Actual vocabulary size    : 32,000
Special tokens            : ['<pad>', '<s>', '</s>', '<unk>']


In [11]:
# ============================================================
# STEP 10: SAVE TRAINED BPE TOKENIZER
# Save vocabulary and merge rules for later reuse
# ============================================================

TOKENIZER_SAVE_DIR = "/kaggle/working/bpe_tokenizer"

os.makedirs(
    TOKENIZER_SAVE_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Save tokenizer files
# ------------------------------------------------------------

tokenizer.save_model(
    TOKENIZER_SAVE_DIR
)


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

print("=" * 70)
print("BPE TOKENIZER SAVED")
print("=" * 70)

print(f"Save directory : {TOKENIZER_SAVE_DIR}")

print("\nSaved files:")

for file_name in sorted(os.listdir(TOKENIZER_SAVE_DIR)):
    file_path = os.path.join(
        TOKENIZER_SAVE_DIR,
        file_name
    )

    print(
        f"{file_name:<20} "
        f"{os.path.getsize(file_path) / 1024:.2f} KB"
    )

BPE TOKENIZER SAVED
Save directory : /kaggle/working/bpe_tokenizer

Saved files:
merges.txt           291.50 KB
vocab.json           500.65 KB


In [12]:
# ============================================================
# STEP 11: RELOAD AND TEST BPE TOKENIZER
# Verify encoding and decoding before dataset preprocessing
# ============================================================

from tokenizers import ByteLevelBPETokenizer


# ------------------------------------------------------------
# Reload saved tokenizer
# ------------------------------------------------------------

bpe_tokenizer = ByteLevelBPETokenizer(
    os.path.join(TOKENIZER_SAVE_DIR, "vocab.json"),
    os.path.join(TOKENIZER_SAVE_DIR, "merges.txt")
)


# ------------------------------------------------------------
# Test encoding and decoding
# ------------------------------------------------------------

sample_text = (
    "The Supreme Court considered the petition "
    "and delivered its judgment."
)

encoded = bpe_tokenizer.encode(sample_text)
decoded = bpe_tokenizer.decode(encoded.ids)


print("=" * 70)
print("BPE TOKENIZER TEST")
print("=" * 70)

print("\nOriginal text:")
print(sample_text)

print("\nToken IDs:")
print(encoded.ids)

print("\nTokens:")
print(encoded.tokens)

print("\nDecoded text:")
print(decoded)

print("\nNumber of tokens:", len(encoded.ids))
print("Vocabulary size :", bpe_tokenizer.get_vocab_size())

BPE TOKENIZER TEST

Original text:
The Supreme Court considered the petition and delivered its judgment.

Token IDs:
[712, 1794, 459, 1231, 263, 478, 296, 3716, 629, 1159, 17]

Tokens:
['The', 'ĠSupreme', 'ĠCourt', 'Ġconsidered', 'Ġthe', 'Ġpetition', 'Ġand', 'Ġdelivered', 'Ġits', 'Ġjudgment', '.']

Decoded text:
The Supreme Court considered the petition and delivered its judgment.

Number of tokens: 11
Vocabulary size : 32000


In [13]:
# ============================================================
# STEP 12: DEFINE MODEL TRAINING DATA MIXTURE
# Set the contribution of each legal source
# ============================================================

SOURCE_MIX = {
    "acts": 0.15,
    "high_courts": 0.30,
    "supreme_court": 0.30,
    "tribunals": 0.25,
}


# ------------------------------------------------------------
# Validate mixture
# ------------------------------------------------------------

total_ratio = sum(SOURCE_MIX.values())

if abs(total_ratio - 1.0) > 1e-8:
    raise ValueError("Source mixture must sum to 1.0")


# ------------------------------------------------------------
# Display mixture
# ------------------------------------------------------------

print("=" * 70)
print("MODEL TRAINING SOURCE MIXTURE")
print("=" * 70)

for source_name, ratio in SOURCE_MIX.items():
    print(
        f"{source_name:<15} "
        f"{ratio * 100:>5.1f}%"
    )

print("-" * 70)
print(f"{'Total':<15} {total_ratio * 100:>5.1f}%")

MODEL TRAINING SOURCE MIXTURE
acts             15.0%
high_courts      30.0%
supreme_court    30.0%
tribunals        25.0%
----------------------------------------------------------------------
Total           100.0%


In [14]:
# ============================================================
# STEP 13: DEFINE ENCODER-DECODER PAIR SIZES
# ============================================================

PROMPT_WORDS = 512
TARGET_WORDS = 128

WORDS_PER_PAIR = PROMPT_WORDS + TARGET_WORDS


print("=" * 70)
print("ENCODER-DECODER PAIR CONFIGURATION")
print("=" * 70)

print(f"Encoder prompt words : {PROMPT_WORDS}")
print(f"Decoder target words : {TARGET_WORDS}")
print(f"Words per pair       : {WORDS_PER_PAIR}")

print("\nPair structure:")
print("First 512 words  -> Encoder input")
print("Next 128 words   -> Decoder target")

ENCODER-DECODER PAIR CONFIGURATION
Encoder prompt words : 512
Decoder target words : 128
Words per pair       : 640

Pair structure:
First 512 words  -> Encoder input
Next 128 words   -> Decoder target


In [15]:
# ============================================================
# STEP 14: MEASURE BPE TOKEN LENGTHS
# Estimate token lengths for 512-word prompts and 128-word targets
# ============================================================

import numpy as np


SAMPLES_PER_SOURCE = 50

encoder_token_lengths = []
decoder_token_lengths = []


print("=" * 70)
print("BPE TOKEN LENGTH ANALYSIS")
print("=" * 70)


for source_name, source_url in TRAIN_SOURCES.items():

    source_samples = 0

    print(f"\nAnalyzing: {source_name}")

    for document in stream_json_documents(
        source_url,
        hf_token
    ):

        text = document.get("text", "")

        if not isinstance(text, str):
            continue

        words = text.split()

        # Need at least one complete 512 + 128 pair
        if len(words) < WORDS_PER_PAIR:
            continue

        encoder_text = " ".join(
            words[:PROMPT_WORDS]
        )

        decoder_text = " ".join(
            words[PROMPT_WORDS:WORDS_PER_PAIR]
        )

        encoder_ids = bpe_tokenizer.encode(
            encoder_text
        ).ids

        decoder_ids = bpe_tokenizer.encode(
            decoder_text
        ).ids

        encoder_token_lengths.append(
            len(encoder_ids)
        )

        decoder_token_lengths.append(
            len(decoder_ids)
        )

        source_samples += 1

        if source_samples >= SAMPLES_PER_SOURCE:
            break

    print(f"Samples collected: {source_samples}")


# ------------------------------------------------------------
# Display token-length statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOKEN LENGTH STATISTICS")
print("=" * 70)

print("\nEncoder: 512 words")
print(f"Mean : {np.mean(encoder_token_lengths):.1f}")
print(f"P95  : {np.percentile(encoder_token_lengths, 95):.1f}")
print(f"Max  : {np.max(encoder_token_lengths)}")

print("\nDecoder: 128 words")
print(f"Mean : {np.mean(decoder_token_lengths):.1f}")
print(f"P95  : {np.percentile(decoder_token_lengths, 95):.1f}")
print(f"Max  : {np.max(decoder_token_lengths)}")

print(
    f"\nTotal samples analyzed: "
    f"{len(encoder_token_lengths)}"
)

BPE TOKEN LENGTH ANALYSIS

Analyzing: acts
Samples collected: 50

Analyzing: high_courts
Samples collected: 50

Analyzing: supreme_court
Samples collected: 50

Analyzing: tribunals
Samples collected: 50

TOKEN LENGTH STATISTICS

Encoder: 512 words
Mean : 661.3
P95  : 746.0
Max  : 1036

Decoder: 128 words
Mean : 158.2
P95  : 190.0
Max  : 252

Total samples analyzed: 200


In [16]:
# ============================================================
# STEP 15: LARGER TOKEN LENGTH ANALYSIS
# Measure the distribution using 1000 legal prompt-target pairs
# ============================================================

SAMPLES_PER_SOURCE = 250

encoder_lengths_large = []
decoder_lengths_large = []


print("=" * 70)
print("LARGE TOKEN LENGTH ANALYSIS")
print("=" * 70)


for source_name, source_url in TRAIN_SOURCES.items():

    source_samples = 0

    print(f"\nAnalyzing: {source_name}")

    for document in stream_json_documents(
        source_url,
        hf_token
    ):

        text = document.get("text", "")

        if not isinstance(text, str):
            continue

        words = text.split()

        if len(words) < WORDS_PER_PAIR:
            continue

        encoder_text = " ".join(
            words[:PROMPT_WORDS]
        )

        decoder_text = " ".join(
            words[PROMPT_WORDS:WORDS_PER_PAIR]
        )

        encoder_ids = bpe_tokenizer.encode(
            encoder_text
        ).ids

        decoder_ids = bpe_tokenizer.encode(
            decoder_text
        ).ids

        encoder_lengths_large.append(
            len(encoder_ids)
        )

        decoder_lengths_large.append(
            len(decoder_ids)
        )

        source_samples += 1

        if source_samples >= SAMPLES_PER_SOURCE:
            break

    print(f"Samples collected: {source_samples}")


# ------------------------------------------------------------
# Display distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ENCODER TOKEN LENGTHS")
print("=" * 70)

for p in [50, 90, 95, 99, 100]:
    print(
        f"P{p:<3}: "
        f"{np.percentile(encoder_lengths_large, p):.0f}"
    )


print("\n" + "=" * 70)
print("DECODER TOKEN LENGTHS")
print("=" * 70)

for p in [50, 90, 95, 99, 100]:
    print(
        f"P{p:<3}: "
        f"{np.percentile(decoder_lengths_large, p):.0f}"
    )


print(
    f"\nTotal pairs analyzed: "
    f"{len(encoder_lengths_large):,}"
)

LARGE TOKEN LENGTH ANALYSIS

Analyzing: acts
Samples collected: 250

Analyzing: high_courts
Samples collected: 250

Analyzing: supreme_court
Samples collected: 250

Analyzing: tribunals
Samples collected: 250

ENCODER TOKEN LENGTHS
P50 : 655
P90 : 726
P95 : 747
P99 : 834
P100: 1776

DECODER TOKEN LENGTHS
P50 : 154
P90 : 180
P95 : 191
P99 : 223
P100: 297

Total pairs analyzed: 1,000


In [17]:
# ============================================================
# STEP 16: DEFINE MODEL SEQUENCE LENGTHS
# Set encoder and decoder token limits and check coverage
# ============================================================

ENCODER_MAX_LENGTH = 1024
DECODER_MAX_LENGTH = 256


# ------------------------------------------------------------
# Measure sample coverage
# ------------------------------------------------------------

encoder_overflow = sum(
    length > ENCODER_MAX_LENGTH
    for length in encoder_lengths_large
)

decoder_overflow = sum(
    length > DECODER_MAX_LENGTH
    for length in decoder_lengths_large
)

total_samples = len(encoder_lengths_large)


print("=" * 70)
print("MODEL SEQUENCE LENGTH CONFIGURATION")
print("=" * 70)

print(f"Encoder max length : {ENCODER_MAX_LENGTH}")
print(f"Decoder max length : {DECODER_MAX_LENGTH}")

print("\nObserved overflow in 1,000-pair sample:")

print(
    f"Encoder : {encoder_overflow:,} / {total_samples:,} "
    f"({100 * encoder_overflow / total_samples:.2f}%)"
)

print(
    f"Decoder : {decoder_overflow:,} / {total_samples:,} "
    f"({100 * decoder_overflow / total_samples:.2f}%)"
)

MODEL SEQUENCE LENGTH CONFIGURATION
Encoder max length : 1024
Decoder max length : 256

Observed overflow in 1,000-pair sample:
Encoder : 2 / 1,000 (0.20%)
Decoder : 3 / 1,000 (0.30%)


In [18]:
# ============================================================
# STEP 17: GET SPECIAL TOKEN IDS
# Retrieve IDs required for encoder-decoder preprocessing
# ============================================================

PAD_TOKEN = "<pad>"
BOS_TOKEN = "<s>"
EOS_TOKEN = "</s>"
UNK_TOKEN = "<unk>"


# ------------------------------------------------------------
# Get token IDs
# ------------------------------------------------------------

PAD_TOKEN_ID = bpe_tokenizer.token_to_id(PAD_TOKEN)
BOS_TOKEN_ID = bpe_tokenizer.token_to_id(BOS_TOKEN)
EOS_TOKEN_ID = bpe_tokenizer.token_to_id(EOS_TOKEN)
UNK_TOKEN_ID = bpe_tokenizer.token_to_id(UNK_TOKEN)


# ------------------------------------------------------------
# Verify IDs
# ------------------------------------------------------------

special_token_ids = {
    PAD_TOKEN: PAD_TOKEN_ID,
    BOS_TOKEN: BOS_TOKEN_ID,
    EOS_TOKEN: EOS_TOKEN_ID,
    UNK_TOKEN: UNK_TOKEN_ID,
}

print("=" * 70)
print("SPECIAL TOKEN IDS")
print("=" * 70)

for token, token_id in special_token_ids.items():
    print(f"{token:<8} -> {token_id}")

if any(token_id is None for token_id in special_token_ids.values()):
    raise ValueError("One or more special tokens are missing.")

print("\nAll special token IDs found successfully.")

SPECIAL TOKEN IDS
<pad>    -> 0
<s>      -> 1
</s>     -> 2
<unk>    -> 3

All special token IDs found successfully.


In [19]:
# ============================================================
# STEP 18: CREATE ONE ENCODER-DECODER PAIR
# Convert legal text into encoder input, decoder input and labels
# ============================================================

def create_training_pair(text):

    words = text.split()

    # Need at least 512 prompt + 128 target words
    if len(words) < WORDS_PER_PAIR:
        return None

    # --------------------------------------------------------
    # Split text at word level
    # --------------------------------------------------------

    encoder_text = " ".join(
        words[:PROMPT_WORDS]
    )

    target_text = " ".join(
        words[PROMPT_WORDS:WORDS_PER_PAIR]
    )

    # --------------------------------------------------------
    # Convert words to BPE token IDs
    # --------------------------------------------------------

    encoder_ids = bpe_tokenizer.encode(
        encoder_text
    ).ids

    target_ids = bpe_tokenizer.encode(
        target_text
    ).ids

    # Skip rare examples that exceed our limits
    if len(encoder_ids) > ENCODER_MAX_LENGTH:
        return None

    # Reserve one decoder position for BOS / EOS
    if len(target_ids) + 1 > DECODER_MAX_LENGTH:
        return None

    # --------------------------------------------------------
    # Teacher-forcing sequences
    # --------------------------------------------------------

    decoder_input_ids = (
        [BOS_TOKEN_ID] + target_ids
    )

    labels = (
        target_ids + [EOS_TOKEN_ID]
    )

    return {
        "encoder_input_ids": encoder_ids,
        "decoder_input_ids": decoder_input_ids,
        "labels": labels,
    }


print("=" * 70)
print("TRAINING PAIR FUNCTION")
print("=" * 70)
print("Training pair function created successfully.")

TRAINING PAIR FUNCTION
Training pair function created successfully.


In [20]:
# ============================================================
# STEP 19: TEST ONE REAL TRAINING PAIR
# Verify encoder input, decoder input and labels
# ============================================================

sample_pair = None


# ------------------------------------------------------------
# Find one valid example
# ------------------------------------------------------------

for document in stream_json_documents(
    TRAIN_SOURCES["supreme_court"],
    hf_token
):

    text = document.get("text", "")

    if not isinstance(text, str):
        continue

    sample_pair = create_training_pair(text)

    if sample_pair is not None:
        break


# ------------------------------------------------------------
# Inspect pair
# ------------------------------------------------------------

print("=" * 70)
print("REAL TRAINING PAIR TEST")
print("=" * 70)

print(
    "Encoder tokens      :",
    len(sample_pair["encoder_input_ids"])
)

print(
    "Decoder input tokens:",
    len(sample_pair["decoder_input_ids"])
)

print(
    "Label tokens        :",
    len(sample_pair["labels"])
)


# ------------------------------------------------------------
# Verify decoder shift
# ------------------------------------------------------------

print("\nFirst 10 decoder input IDs:")
print(sample_pair["decoder_input_ids"][:10])

print("\nFirst 10 label IDs:")
print(sample_pair["labels"][:10])

print("\nLast decoder input ID:")
print(sample_pair["decoder_input_ids"][-1])

print("\nLast label ID:")
print(sample_pair["labels"][-1])


# ------------------------------------------------------------
# Decode for visual verification
# ------------------------------------------------------------

encoder_preview = bpe_tokenizer.decode(
    sample_pair["encoder_input_ids"]
)

target_preview = bpe_tokenizer.decode(
    sample_pair["labels"][:-1]
)

print("\nEncoder text preview:")
print(encoder_preview[:500])

print("\nTarget text preview:")
print(target_preview[:300])

REAL TRAINING PAIR TEST
Encoder tokens      : 697
Decoder input tokens: 151
Label tokens        : 151

First 10 decoder input IDs:
[1, 2405, 274, 17, 694, 270, 263, 436, 15, 1408]

First 10 label IDs:
[2405, 274, 17, 694, 270, 263, 436, 15, 1408, 337]

Last decoder input ID:
263

Last label ID:
2

Encoder text preview:
PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs. RESPONDENT: THE STATE OF MYSORE DATE OF JUDGMENT: 18/12/1961 ACT: Temple-Large number of pilgrims visiting and worshiping-If private or public-Madras Temple Entry Authorisation Act, 1947(Mad. V. of 1947), s. 2(1). HEADNOTE: The shrine in suit which was originally founded by Shri Madhavacharya, a Hindu saint was managed for a long time in rotation by the heads of eight Mutts which were also founded by the said saint for eigh

Target text preview:
under s. 6 of the Act, decided on June 3, 1948, that Shri Krishna Mutt was a temple. The plaintiffs, who represent the heads of six of the eight mutts estab

In [21]:
# ============================================================
# STEP 20: CREATE MULTIPLE PAIRS PER DOCUMENT
# Convert long legal documents into consecutive training pairs
# ============================================================

def create_training_pairs(text):

    words = text.split()

    pairs = []

    # --------------------------------------------------------
    # Process consecutive 640-word sections
    # --------------------------------------------------------

    for start in range(
        0,
        len(words) - WORDS_PER_PAIR + 1,
        WORDS_PER_PAIR
    ):

        encoder_words = words[
            start : start + PROMPT_WORDS
        ]

        target_words = words[
            start + PROMPT_WORDS :
            start + WORDS_PER_PAIR
        ]

        encoder_text = " ".join(encoder_words)
        target_text = " ".join(target_words)

        # ----------------------------------------------------
        # BPE tokenization
        # ----------------------------------------------------

        encoder_ids = bpe_tokenizer.encode(
            encoder_text
        ).ids

        target_ids = bpe_tokenizer.encode(
            target_text
        ).ids

        # Skip sequences exceeding model limits
        if len(encoder_ids) > ENCODER_MAX_LENGTH:
            continue

        if len(target_ids) + 1 > DECODER_MAX_LENGTH:
            continue

        # ----------------------------------------------------
        # Build encoder-decoder example
        # ----------------------------------------------------

        pair = {
            "encoder_input_ids": encoder_ids,
            "decoder_input_ids": [
                BOS_TOKEN_ID
            ] + target_ids,
            "labels": target_ids + [
                EOS_TOKEN_ID
            ],
        }

        pairs.append(pair)

    return pairs


print("=" * 70)
print("MULTI-PAIR FUNCTION")
print("=" * 70)
print("Multiple-pair function created successfully.")

MULTI-PAIR FUNCTION
Multiple-pair function created successfully.


In [22]:
# ============================================================
# STEP 21: TEST MULTIPLE TRAINING PAIRS
# Verify that one long document produces consecutive pairs
# ============================================================

test_document = None
test_pairs = None


# ------------------------------------------------------------
# Find one long legal document
# ------------------------------------------------------------

for document in stream_json_documents(
    TRAIN_SOURCES["supreme_court"],
    hf_token
):

    text = document.get("text", "")

    if not isinstance(text, str):
        continue

    # Look for a document long enough for multiple pairs
    if len(text.split()) < WORDS_PER_PAIR * 3:
        continue

    pairs = create_training_pairs(text)

    if len(pairs) >= 3:
        test_document = document
        test_pairs = pairs
        break


# ------------------------------------------------------------
# Verify generated pairs
# ------------------------------------------------------------

print("=" * 70)
print("MULTIPLE TRAINING PAIR TEST")
print("=" * 70)

print(
    f"Document words       : "
    f"{len(test_document['text'].split()):,}"
)

print(
    f"Training pairs       : "
    f"{len(test_pairs):,}"
)


print("\nFirst 3 pairs:")

for i, pair in enumerate(test_pairs[:3], start=1):

    print(f"\nPair {i}")

    print(
        f"Encoder tokens       : "
        f"{len(pair['encoder_input_ids'])}"
    )

    print(
        f"Decoder input tokens : "
        f"{len(pair['decoder_input_ids'])}"
    )

    print(
        f"Label tokens         : "
        f"{len(pair['labels'])}"
    )

    print(
        f"Decoder starts BOS   : "
        f"{pair['decoder_input_ids'][0] == BOS_TOKEN_ID}"
    )

    print(
        f"Labels end EOS       : "
        f"{pair['labels'][-1] == EOS_TOKEN_ID}"
    )

MULTIPLE TRAINING PAIR TEST
Document words       : 4,269
Training pairs       : 6

First 3 pairs:

Pair 1
Encoder tokens       : 697
Decoder input tokens : 151
Label tokens         : 151
Decoder starts BOS   : True
Labels end EOS       : True

Pair 2
Encoder tokens       : 635
Decoder input tokens : 151
Label tokens         : 151
Decoder starts BOS   : True
Labels end EOS       : True

Pair 3
Encoder tokens       : 643
Decoder input tokens : 144
Label tokens         : 144
Decoder starts BOS   : True
Labels end EOS       : True


In [23]:
# ============================================================
# STEP 22: DEFINE TRAINING TOKEN BUDGET
# Allocate 200M training tokens across the four legal sources
# ============================================================

TOTAL_TRAIN_TOKENS = 200_000_000


# ------------------------------------------------------------
# Calculate token budget for each source
# ------------------------------------------------------------

SOURCE_TOKEN_BUDGETS = {
    source: int(TOTAL_TRAIN_TOKENS * ratio)
    for source, ratio in SOURCE_MIX.items()
}


# ------------------------------------------------------------
# Display allocation
# ------------------------------------------------------------

print("=" * 70)
print("MODEL TRAINING TOKEN BUDGET")
print("=" * 70)

for source, token_budget in SOURCE_TOKEN_BUDGETS.items():

    print(
        f"{source:<15} "
        f"{token_budget:>12,} tokens "
        f"({SOURCE_MIX[source] * 100:.1f}%)"
    )


print("-" * 70)

print(
    f"{'Total':<15} "
    f"{sum(SOURCE_TOKEN_BUDGETS.values()):>12,} tokens"
)

MODEL TRAINING TOKEN BUDGET
acts              30,000,000 tokens (15.0%)
high_courts       60,000,000 tokens (30.0%)
supreme_court     60,000,000 tokens (30.0%)
tribunals         50,000,000 tokens (25.0%)
----------------------------------------------------------------------
Total            200,000,000 tokens


In [24]:
# ============================================================
# STEP 23: SET UP PROCESSED DATASET STORAGE
# Save training pairs in shards instead of keeping them in RAM
# ============================================================

import os


PROCESSED_DATA_DIR = (
    "/kaggle/working/arch2_processed_data"
)

os.makedirs(
    PROCESSED_DATA_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Shard configuration
# ------------------------------------------------------------

PAIRS_PER_SHARD = 10_000


print("=" * 70)
print("PROCESSED DATASET STORAGE")
print("=" * 70)

print(f"Output directory : {PROCESSED_DATA_DIR}")
print(f"Pairs per shard  : {PAIRS_PER_SHARD:,}")

print("\nStorage strategy:")
print("Stream document -> create pairs -> save shard -> continue")

PROCESSED DATASET STORAGE
Output directory : /kaggle/working/arch2_processed_data
Pairs per shard  : 10,000

Storage strategy:
Stream document -> create pairs -> save shard -> continue


In [25]:
# ============================================================
# STEP 24: CREATE SHARD SAVING FUNCTION
# Pad training pairs and save them efficiently as NumPy shards
# ============================================================

import numpy as np


def save_training_shard(pairs, shard_path):

    num_pairs = len(pairs)

    # --------------------------------------------------------
    # Allocate fixed-size arrays
    # --------------------------------------------------------

    encoder_input_ids = np.full(
        (num_pairs, ENCODER_MAX_LENGTH),
        PAD_TOKEN_ID,
        dtype=np.int32
    )

    decoder_input_ids = np.full(
        (num_pairs, DECODER_MAX_LENGTH),
        PAD_TOKEN_ID,
        dtype=np.int32
    )

    labels = np.full(
        (num_pairs, DECODER_MAX_LENGTH),
        -100,
        dtype=np.int32
    )


    # --------------------------------------------------------
    # Fill arrays with actual token IDs
    # --------------------------------------------------------

    for i, pair in enumerate(pairs):

        enc = pair["encoder_input_ids"]
        dec = pair["decoder_input_ids"]
        lab = pair["labels"]

        encoder_input_ids[i, :len(enc)] = enc
        decoder_input_ids[i, :len(dec)] = dec
        labels[i, :len(lab)] = lab


    # --------------------------------------------------------
    # Save shard
    # --------------------------------------------------------

    np.savez_compressed(
        shard_path,
        encoder_input_ids=encoder_input_ids,
        decoder_input_ids=decoder_input_ids,
        labels=labels
    )


print("=" * 70)
print("SHARD SAVING FUNCTION")
print("=" * 70)
print("Shard saving function created successfully.")

SHARD SAVING FUNCTION
Shard saving function created successfully.


In [26]:
# ============================================================
# STEP 25: TEST SHARD SAVE AND RELOAD
# Verify padded arrays before large-scale preprocessing
# ============================================================

TEST_SHARD_PATH = os.path.join(
    PROCESSED_DATA_DIR,
    "test_shard.npz"
)


# ------------------------------------------------------------
# Save test pairs
# ------------------------------------------------------------

save_training_shard(
    test_pairs,
    TEST_SHARD_PATH
)


# ------------------------------------------------------------
# Reload saved shard
# ------------------------------------------------------------

test_shard = np.load(TEST_SHARD_PATH)

encoder_array = test_shard["encoder_input_ids"]
decoder_array = test_shard["decoder_input_ids"]
labels_array = test_shard["labels"]


# ------------------------------------------------------------
# Verify shapes and special values
# ------------------------------------------------------------

print("=" * 70)
print("TEST SHARD VERIFICATION")
print("=" * 70)

print(f"Number of pairs : {encoder_array.shape[0]:,}")

print("\nArray shapes:")
print("Encoder :", encoder_array.shape)
print("Decoder :", decoder_array.shape)
print("Labels  :", labels_array.shape)

print("\nData types:")
print("Encoder :", encoder_array.dtype)
print("Decoder :", decoder_array.dtype)
print("Labels  :", labels_array.dtype)

print("\nFirst example checks:")
print("Decoder starts BOS :", decoder_array[0, 0] == BOS_TOKEN_ID)

real_label_positions = np.where(
    labels_array[0] != -100
)[0]

last_real_label = labels_array[
    0,
    real_label_positions[-1]
]

print("Labels end EOS     :", last_real_label == EOS_TOKEN_ID)
print("Encoder has padding:", np.any(encoder_array[0] == PAD_TOKEN_ID))
print("Labels have -100   :", np.any(labels_array[0] == -100))

print(
    "\nShard size         :",
    f"{os.path.getsize(TEST_SHARD_PATH) / (1024**2):.2f} MB"
)

TEST SHARD VERIFICATION
Number of pairs : 6

Array shapes:
Encoder : (6, 1024)
Decoder : (6, 256)
Labels  : (6, 256)

Data types:
Encoder : int32
Decoder : int32
Labels  : int32

First example checks:
Decoder starts BOS : True
Labels end EOS     : True
Encoder has padding: True
Labels have -100   : True

Shard size         : 0.01 MB


In [27]:
# ============================================================
# STEP 26: CREATE SOURCE PREPROCESSING FUNCTION
# Stream one source, create pairs and save training shards
# ============================================================

def preprocess_source(
    source_name,
    source_url,
    token_budget,
    output_dir,
    pairs_per_shard=PAIRS_PER_SHARD
):

    source_dir = os.path.join(
        output_dir,
        source_name
    )

    os.makedirs(
        source_dir,
        exist_ok=True
    )

    shard_pairs = []

    shard_index = 0
    total_pairs = 0
    total_tokens = 0
    total_documents = 0
    skipped_pairs = 0


    print("=" * 70)
    print(f"PROCESSING SOURCE: {source_name}")
    print("=" * 70)

    print(f"Token budget: {token_budget:,}")


    # --------------------------------------------------------
    # Stream documents
    # --------------------------------------------------------

    for document in stream_json_documents(
        source_url,
        hf_token
    ):

        text = document.get("text", "")

        if not isinstance(text, str):
            continue

        text = text.strip()

        if not text:
            continue

        total_documents += 1

        words = text.split()


        # ----------------------------------------------------
        # Create consecutive 640-word sections
        # ----------------------------------------------------

        for start in range(
            0,
            len(words) - WORDS_PER_PAIR + 1,
            WORDS_PER_PAIR
        ):

            encoder_text = " ".join(
                words[
                    start :
                    start + PROMPT_WORDS
                ]
            )

            target_text = " ".join(
                words[
                    start + PROMPT_WORDS :
                    start + WORDS_PER_PAIR
                ]
            )


            # ------------------------------------------------
            # BPE tokenization
            # ------------------------------------------------

            encoder_ids = bpe_tokenizer.encode(
                encoder_text
            ).ids

            target_ids = bpe_tokenizer.encode(
                target_text
            ).ids


            # ------------------------------------------------
            # Skip sequences exceeding model limits
            # ------------------------------------------------

            if len(encoder_ids) > ENCODER_MAX_LENGTH:
                skipped_pairs += 1
                continue

            if len(target_ids) + 1 > DECODER_MAX_LENGTH:
                skipped_pairs += 1
                continue


            pair_tokens = (
                len(encoder_ids) +
                len(target_ids)
            )


            # Stop before exceeding source budget
            if total_tokens + pair_tokens > token_budget:
                break


            # ------------------------------------------------
            # Build training pair
            # ------------------------------------------------

            pair = {
                "encoder_input_ids": encoder_ids,

                "decoder_input_ids": (
                    [BOS_TOKEN_ID] +
                    target_ids
                ),

                "labels": (
                    target_ids +
                    [EOS_TOKEN_ID]
                )
            }


            shard_pairs.append(pair)

            total_pairs += 1
            total_tokens += pair_tokens


            # ------------------------------------------------
            # Save full shard
            # ------------------------------------------------

            if len(shard_pairs) >= pairs_per_shard:

                shard_path = os.path.join(
                    source_dir,
                    f"shard_{shard_index:05d}.npz"
                )

                save_training_shard(
                    shard_pairs,
                    shard_path
                )

                print(
                    f"Saved shard {shard_index:05d} | "
                    f"pairs={total_pairs:,} | "
                    f"tokens={total_tokens:,}"
                )

                shard_pairs = []
                shard_index += 1


        # ----------------------------------------------------
        # Stop when source budget is reached
        # ----------------------------------------------------

        if total_tokens >= token_budget:
            break


    # --------------------------------------------------------
    # Save final partial shard
    # --------------------------------------------------------

    if shard_pairs:

        shard_path = os.path.join(
            source_dir,
            f"shard_{shard_index:05d}.npz"
        )

        save_training_shard(
            shard_pairs,
            shard_path
        )

        shard_index += 1


    # --------------------------------------------------------
    # Final source statistics
    # --------------------------------------------------------

    stats = {
        "source": source_name,
        "documents": total_documents,
        "pairs": total_pairs,
        "tokens": total_tokens,
        "skipped_pairs": skipped_pairs,
        "shards": shard_index,
    }

    return stats


print("=" * 70)
print("SOURCE PREPROCESSING FUNCTION")
print("=" * 70)
print("Preprocessing function created successfully.")

SOURCE PREPROCESSING FUNCTION
Preprocessing function created successfully.


In [28]:
# ============================================================
# STEP 27: SMALL PREPROCESSING DRY RUN
# Test the complete pipeline before processing 200M tokens
# ============================================================

TEST_PREPROCESS_DIR = (
    "/kaggle/working/arch2_preprocess_test"
)

os.makedirs(
    TEST_PREPROCESS_DIR,
    exist_ok=True
)

TEST_TOKEN_BUDGET = 100_000


# ------------------------------------------------------------
# Run small test
# ------------------------------------------------------------

test_stats = preprocess_source(
    source_name="acts",
    source_url=TRAIN_SOURCES["acts"],
    token_budget=TEST_TOKEN_BUDGET,
    output_dir=TEST_PREPROCESS_DIR,
    pairs_per_shard=100
)


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DRY RUN COMPLETE")
print("=" * 70)

for key, value in test_stats.items():

    if isinstance(value, int):
        print(f"{key:<15}: {value:,}")
    else:
        print(f"{key:<15}: {value}")

PROCESSING SOURCE: acts
Token budget: 100,000
Saved shard 00000 | pairs=100 | tokens=76,088

DRY RUN COMPLETE
source         : acts
documents      : 18,822
pairs          : 131
tokens         : 99,904
skipped_pairs  : 80
shards         : 2


In [29]:
# ============================================================
# STEP 28: VERIFY DRY-RUN SHARDS
# Check saved shard files, shapes and total training pairs
# ============================================================

import glob
import os
import numpy as np


test_shard_files = sorted(
    glob.glob(
        os.path.join(
            TEST_PREPROCESS_DIR,
            "acts",
            "shard_*.npz"
        )
    )
)


print("=" * 70)
print("DRY-RUN SHARD VERIFICATION")
print("=" * 70)

print(f"Shard files found : {len(test_shard_files)}")


total_saved_pairs = 0


for shard_path in test_shard_files:

    with np.load(shard_path) as shard:

        encoder = shard["encoder_input_ids"]
        decoder = shard["decoder_input_ids"]
        labels = shard["labels"]

        num_pairs = encoder.shape[0]

        total_saved_pairs += num_pairs

        print(
            f"\n{os.path.basename(shard_path)}"
        )

        print(
            f"Pairs   : {num_pairs}"
        )

        print(
            f"Encoder : {encoder.shape}"
        )

        print(
            f"Decoder : {decoder.shape}"
        )

        print(
            f"Labels  : {labels.shape}"
        )


print("\n" + "-" * 70)

print(
    f"Total saved pairs : {total_saved_pairs:,}"
)

print(
    f"Expected pairs    : {test_stats['pairs']:,}"
)

print(
    "Pair count match  :",
    total_saved_pairs == test_stats["pairs"]
)

DRY-RUN SHARD VERIFICATION
Shard files found : 2

shard_00000.npz
Pairs   : 100
Encoder : (100, 1024)
Decoder : (100, 256)
Labels  : (100, 256)

shard_00001.npz
Pairs   : 31
Encoder : (31, 1024)
Decoder : (31, 256)
Labels  : (31, 256)

----------------------------------------------------------------------
Total saved pairs : 131
Expected pairs    : 131
Pair count match  : True


In [30]:
# ============================================================
# STEP 29: DEFINE TRAIN-VALIDATION SPLIT
# Reserve documents for validation before pair generation
# ============================================================

VALIDATION_RATIO = 0.01

SPLIT_SEED = 42


print("=" * 70)
print("TRAIN-VALIDATION SPLIT CONFIGURATION")
print("=" * 70)

print(
    f"Training ratio   : "
    f"{(1 - VALIDATION_RATIO) * 100:.1f}%"
)

print(
    f"Validation ratio : "
    f"{VALIDATION_RATIO * 100:.1f}%"
)

print(
    f"Split seed       : "
    f"{SPLIT_SEED}"
)

print("\nSplit level:")
print("Document-level")

TRAIN-VALIDATION SPLIT CONFIGURATION
Training ratio   : 99.0%
Validation ratio : 1.0%
Split seed       : 42

Split level:
Document-level


In [31]:
# ============================================================
# STEP 30: CREATE DOCUMENT SPLIT FUNCTION
# Assign each complete document to train or validation
# ============================================================

import hashlib


def get_document_split(
    source_name,
    document,
    validation_ratio=VALIDATION_RATIO
):

    # --------------------------------------------------------
    # Build a stable identifier for the document
    # --------------------------------------------------------

    text = document.get("text", "")

    identifier = (
        source_name
        + "|"
        + str(document.get("title", ""))
        + "|"
        + str(document.get("year", ""))
        + "|"
        + text[:500]
    )


    # --------------------------------------------------------
    # Create deterministic hash
    # --------------------------------------------------------

    hash_value = hashlib.sha256(
        identifier.encode("utf-8")
    ).hexdigest()

    hash_number = int(
        hash_value[:16],
        16
    )

    split_value = (
        hash_number /
        float(0xFFFFFFFFFFFFFFFF)
    )


    # --------------------------------------------------------
    # Assign document
    # --------------------------------------------------------

    if split_value < validation_ratio:
        return "validation"

    return "train"


print("=" * 70)
print("DOCUMENT SPLIT FUNCTION")
print("=" * 70)
print("Deterministic document split function created successfully.")

DOCUMENT SPLIT FUNCTION
Deterministic document split function created successfully.


In [32]:
# ============================================================
# STEP 31: TEST DOCUMENT SPLIT FUNCTION
# Verify deterministic train-validation document assignment
# ============================================================

TEST_DOCUMENTS_PER_SOURCE = 1000


print("=" * 70)
print("DOCUMENT SPLIT TEST")
print("=" * 70)


for source_name, source_url in TRAIN_SOURCES.items():

    train_count = 0
    validation_count = 0
    checked = 0

    first_document = None


    for document in stream_json_documents(
        source_url,
        hf_token
    ):

        text = document.get("text", "")

        if not isinstance(text, str) or not text.strip():
            continue


        if first_document is None:
            first_document = document


        split = get_document_split(
            source_name,
            document
        )


        if split == "train":
            train_count += 1
        else:
            validation_count += 1


        checked += 1

        if checked >= TEST_DOCUMENTS_PER_SOURCE:
            break


    # --------------------------------------------------------
    # Check deterministic behavior
    # --------------------------------------------------------

    split_1 = get_document_split(
        source_name,
        first_document
    )

    split_2 = get_document_split(
        source_name,
        first_document
    )


    print(f"\nSource: {source_name}")

    print(
        f"Train      : {train_count:,}"
    )

    print(
        f"Validation : {validation_count:,}"
    )

    print(
        f"Total      : {checked:,}"
    )

    print(
        "Deterministic:",
        split_1 == split_2
    )

DOCUMENT SPLIT TEST

Source: acts
Train      : 995
Validation : 5
Total      : 1,000
Deterministic: True

Source: high_courts
Train      : 989
Validation : 11
Total      : 1,000
Deterministic: True

Source: supreme_court
Train      : 994
Validation : 6
Total      : 1,000
Deterministic: True

Source: tribunals
Train      : 995
Validation : 5
Total      : 1,000
Deterministic: True


In [33]:
# ============================================================
# STEP 32: CREATE FINAL PREPROCESSING FUNCTION
# Stream documents and save train/validation shards separately
# ============================================================

def preprocess_source_with_split(
    source_name,
    source_url,
    train_token_budget,
    output_dir,
    pairs_per_shard=PAIRS_PER_SHARD
):

    # --------------------------------------------------------
    # Create source directories
    # --------------------------------------------------------

    train_dir = os.path.join(
        output_dir,
        "train",
        source_name
    )

    validation_dir = os.path.join(
        output_dir,
        "validation",
        source_name
    )

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(validation_dir, exist_ok=True)


    # --------------------------------------------------------
    # Processing state
    # --------------------------------------------------------

    train_pairs = []
    validation_pairs = []

    train_shard_index = 0
    validation_shard_index = 0

    total_train_pairs = 0
    total_validation_pairs = 0

    total_train_tokens = 0
    total_validation_tokens = 0

    total_documents = 0
    train_documents = 0
    validation_documents = 0

    skipped_pairs = 0


    print("=" * 70)
    print(f"PROCESSING SOURCE: {source_name}")
    print("=" * 70)

    print(
        f"Training token budget: "
        f"{train_token_budget:,}"
    )


    # ========================================================
    # STREAM DOCUMENTS
    # ========================================================

    for document in stream_json_documents(
        source_url,
        hf_token
    ):

        text = document.get("text", "")

        if not isinstance(text, str):
            continue

        text = text.strip()

        if not text:
            continue


        # ----------------------------------------------------
        # Assign complete document to one split
        # ----------------------------------------------------

        split = get_document_split(
            source_name,
            document
        )

        total_documents += 1

        if split == "train":
            train_documents += 1
        else:
            validation_documents += 1


        words = text.split()


        # ----------------------------------------------------
        # Create consecutive prompt-target pairs
        # ----------------------------------------------------

        for start in range(
            0,
            len(words) - WORDS_PER_PAIR + 1,
            WORDS_PER_PAIR
        ):

            encoder_text = " ".join(
                words[
                    start :
                    start + PROMPT_WORDS
                ]
            )

            target_text = " ".join(
                words[
                    start + PROMPT_WORDS :
                    start + WORDS_PER_PAIR
                ]
            )


            encoder_ids = bpe_tokenizer.encode(
                encoder_text
            ).ids

            target_ids = bpe_tokenizer.encode(
                target_text
            ).ids


            # ------------------------------------------------
            # Reject sequences exceeding model limits
            # ------------------------------------------------

            if len(encoder_ids) > ENCODER_MAX_LENGTH:
                skipped_pairs += 1
                continue

            if len(target_ids) + 1 > DECODER_MAX_LENGTH:
                skipped_pairs += 1
                continue


            pair = {
                "encoder_input_ids": encoder_ids,

                "decoder_input_ids":
                    [BOS_TOKEN_ID] + target_ids,

                "labels":
                    target_ids + [EOS_TOKEN_ID],
            }


            pair_tokens = (
                len(encoder_ids) +
                len(target_ids)
            )


            # =================================================
            # TRAINING PAIR
            # =================================================

            if split == "train":

                if (
                    total_train_tokens + pair_tokens
                    > train_token_budget
                ):
                    break

                train_pairs.append(pair)

                total_train_pairs += 1
                total_train_tokens += pair_tokens


                if len(train_pairs) >= pairs_per_shard:

                    shard_path = os.path.join(
                        train_dir,
                        f"shard_{train_shard_index:05d}.npz"
                    )

                    save_training_shard(
                        train_pairs,
                        shard_path
                    )

                    print(
                        f"Train shard "
                        f"{train_shard_index:05d} | "
                        f"pairs={total_train_pairs:,} | "
                        f"tokens={total_train_tokens:,}"
                    )

                    train_pairs = []
                    train_shard_index += 1


            # =================================================
            # VALIDATION PAIR
            # =================================================

            else:

                validation_pairs.append(pair)

                total_validation_pairs += 1
                total_validation_tokens += pair_tokens


                if len(validation_pairs) >= pairs_per_shard:

                    shard_path = os.path.join(
                        validation_dir,
                        f"shard_{validation_shard_index:05d}.npz"
                    )

                    save_training_shard(
                        validation_pairs,
                        shard_path
                    )

                    validation_pairs = []
                    validation_shard_index += 1


        # ----------------------------------------------------
        # Stop after training budget is reached
        # ----------------------------------------------------

        if total_train_tokens >= train_token_budget:
            break


    # ========================================================
    # SAVE FINAL PARTIAL SHARDS
    # ========================================================

    if train_pairs:

        shard_path = os.path.join(
            train_dir,
            f"shard_{train_shard_index:05d}.npz"
        )

        save_training_shard(
            train_pairs,
            shard_path
        )

        train_shard_index += 1


    if validation_pairs:

        shard_path = os.path.join(
            validation_dir,
            f"shard_{validation_shard_index:05d}.npz"
        )

        save_training_shard(
            validation_pairs,
            shard_path
        )

        validation_shard_index += 1


    # ========================================================
    # RETURN STATISTICS
    # ========================================================

    return {
        "source": source_name,

        "documents": total_documents,
        "train_documents": train_documents,
        "validation_documents": validation_documents,

        "train_pairs": total_train_pairs,
        "validation_pairs": total_validation_pairs,

        "train_tokens": total_train_tokens,
        "validation_tokens": total_validation_tokens,

        "skipped_pairs": skipped_pairs,

        "train_shards": train_shard_index,
        "validation_shards": validation_shard_index,
    }


print("=" * 70)
print("FINAL PREPROCESSING FUNCTION")
print("=" * 70)
print("Train-validation preprocessing function created successfully.")

FINAL PREPROCESSING FUNCTION
Train-validation preprocessing function created successfully.


In [34]:
# ============================================================
# STEP 33: SPLIT-AWARE PREPROCESSING DRY RUN
# Test separate train and validation shard creation
# ============================================================

import shutil


SPLIT_TEST_DIR = (
    "/kaggle/working/arch2_split_test"
)

# Remove old test directory if it exists
if os.path.exists(SPLIT_TEST_DIR):
    shutil.rmtree(SPLIT_TEST_DIR)

os.makedirs(
    SPLIT_TEST_DIR,
    exist_ok=True
)


SPLIT_TEST_TOKEN_BUDGET = 500_000


# ------------------------------------------------------------
# Run split-aware preprocessing
# ------------------------------------------------------------

split_test_stats = preprocess_source_with_split(
    source_name="acts",
    source_url=TRAIN_SOURCES["acts"],
    train_token_budget=SPLIT_TEST_TOKEN_BUDGET,
    output_dir=SPLIT_TEST_DIR,
    pairs_per_shard=500
)


# ------------------------------------------------------------
# Display statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT-AWARE DRY RUN COMPLETE")
print("=" * 70)


for key, value in split_test_stats.items():

    if isinstance(value, int):
        print(f"{key:<22}: {value:,}")
    else:
        print(f"{key:<22}: {value}")

PROCESSING SOURCE: acts
Training token budget: 500,000
Train shard 00000 | pairs=500 | tokens=390,567

SPLIT-AWARE DRY RUN COMPLETE
source                : acts
documents             : 18,822
train_documents       : 18,649
validation_documents  : 173
train_pairs           : 643
validation_pairs      : 1,688
train_tokens          : 499,382
validation_tokens     : 1,328,160
skipped_pairs         : 89
train_shards          : 2
validation_shards     : 4


In [35]:
# ============================================================
# STEP 34: VERIFY SPLIT-AWARE DRY RUN
# Check train and validation shards saved by the final pipeline
# ============================================================

import glob
import os
import numpy as np


train_test_shards = sorted(
    glob.glob(
        os.path.join(
            SPLIT_TEST_DIR,
            "train",
            "acts",
            "shard_*.npz"
        )
    )
)

validation_test_shards = sorted(
    glob.glob(
        os.path.join(
            SPLIT_TEST_DIR,
            "validation",
            "acts",
            "shard_*.npz"
        )
    )
)


def verify_shards(shard_files):

    total_pairs = 0

    for shard_path in shard_files:

        with np.load(shard_path) as shard:

            encoder = shard["encoder_input_ids"]
            decoder = shard["decoder_input_ids"]
            labels = shard["labels"]

            assert encoder.shape[1] == ENCODER_MAX_LENGTH
            assert decoder.shape[1] == DECODER_MAX_LENGTH
            assert labels.shape[1] == DECODER_MAX_LENGTH

            assert encoder.shape[0] == decoder.shape[0]
            assert decoder.shape[0] == labels.shape[0]

            total_pairs += encoder.shape[0]

    return total_pairs


train_saved_pairs = verify_shards(
    train_test_shards
)

validation_saved_pairs = verify_shards(
    validation_test_shards
)


print("=" * 70)
print("SPLIT-AWARE SHARD VERIFICATION")
print("=" * 70)

print("\nTraining:")
print(f"Shards         : {len(train_test_shards)}")
print(f"Saved pairs    : {train_saved_pairs:,}")
print(f"Expected pairs : {split_test_stats['train_pairs']:,}")
print(
    "Match          :",
    train_saved_pairs == split_test_stats["train_pairs"]
)

print("\nValidation:")
print(f"Shards         : {len(validation_test_shards)}")
print(f"Saved pairs    : {validation_saved_pairs:,}")
print(f"Expected pairs : {split_test_stats['validation_pairs']:,}")
print(
    "Match          :",
    validation_saved_pairs
    == split_test_stats["validation_pairs"]
)

SPLIT-AWARE SHARD VERIFICATION

Training:
Shards         : 2
Saved pairs    : 643
Expected pairs : 643
Match          : True

Validation:
Shards         : 4
Saved pairs    : 1,688
Expected pairs : 1,688
Match          : True


In [36]:
# ============================================================
# STEP 35: PREPARE FINAL PREPROCESSING DIRECTORY
# Create a clean location for the production training dataset
# ============================================================

import os
import shutil


FINAL_PROCESSED_DIR = (
    "/kaggle/working/arch2_final_processed"
)


# ------------------------------------------------------------
# Start with a clean production directory
# ------------------------------------------------------------

if os.path.exists(FINAL_PROCESSED_DIR):
    shutil.rmtree(FINAL_PROCESSED_DIR)

os.makedirs(
    FINAL_PROCESSED_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Display final processing plan
# ------------------------------------------------------------

print("=" * 70)
print("FINAL PREPROCESSING DIRECTORY")
print("=" * 70)

print(f"Output directory : {FINAL_PROCESSED_DIR}")

print("\nProduction token budgets:")

for source_name, token_budget in SOURCE_TOKEN_BUDGETS.items():

    print(
        f"{source_name:<15} "
        f"{token_budget:>12,}"
    )


print("-" * 70)

print(
    f"{'Total':<15} "
    f"{sum(SOURCE_TOKEN_BUDGETS.values()):>12,}"
)

print("\nDirectory prepared successfully.")

FINAL PREPROCESSING DIRECTORY
Output directory : /kaggle/working/arch2_final_processed

Production token budgets:
acts              30,000,000
high_courts       60,000,000
supreme_court     60,000,000
tribunals         50,000,000
----------------------------------------------------------------------
Total            200,000,000

Directory prepared successfully.


In [37]:
# ============================================================
# STEP 36: PROCESS ACTS SOURCE
# Build final train and validation shards for Acts
# ============================================================

acts_stats = preprocess_source_with_split(
    source_name="acts",
    source_url=TRAIN_SOURCES["acts"],
    train_token_budget=SOURCE_TOKEN_BUDGETS["acts"],
    output_dir=FINAL_PROCESSED_DIR,
    pairs_per_shard=PAIRS_PER_SHARD
)


# ------------------------------------------------------------
# Display final Acts statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTS PREPROCESSING COMPLETE")
print("=" * 70)


for key, value in acts_stats.items():

    if isinstance(value, int):
        print(f"{key:<22}: {value:,}")
    else:
        print(f"{key:<22}: {value}")

PROCESSING SOURCE: acts
Training token budget: 30,000,000
Train shard 00000 | pairs=10,000 | tokens=7,922,190
Train shard 00001 | pairs=20,000 | tokens=15,922,583
Train shard 00002 | pairs=30,000 | tokens=23,895,919

ACTS PREPROCESSING COMPLETE
source                : acts
documents             : 18,822
train_documents       : 18,649
validation_documents  : 173
train_pairs           : 37,717
validation_pairs      : 1,688
train_tokens          : 29,999,820
validation_tokens     : 1,328,160
skipped_pairs         : 487
train_shards          : 4
validation_shards     : 1


In [38]:
# ============================================================
# STEP 37: SAVE PREPROCESSING STATE
# Record completed sources for safe restart and reuse
# ============================================================

import json
import os


PREPROCESSING_STATE_PATH = os.path.join(
    FINAL_PROCESSED_DIR,
    "preprocessing_state.json"
)


preprocessing_state = {
    "total_train_token_budget": TOTAL_TRAIN_TOKENS,

    "encoder_max_length": ENCODER_MAX_LENGTH,
    "decoder_max_length": DECODER_MAX_LENGTH,

    "prompt_words": PROMPT_WORDS,
    "target_words": TARGET_WORDS,

    "pairs_per_shard": PAIRS_PER_SHARD,

    "sources": {
        "acts": acts_stats
    }
}


with open(
    PREPROCESSING_STATE_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        preprocessing_state,
        file,
        indent=4
    )


print("=" * 70)
print("PREPROCESSING STATE SAVED")
print("=" * 70)

print(
    f"State file : "
    f"{PREPROCESSING_STATE_PATH}"
)

print("\nCompleted sources:")

for source_name in preprocessing_state["sources"]:
    print(
        f"{source_name:<15} DONE"
    )

print(
    "\nRecorded Acts tokens:",
    f"{acts_stats['train_tokens']:,}"
)

PREPROCESSING STATE SAVED
State file : /kaggle/working/arch2_final_processed/preprocessing_state.json

Completed sources:
acts            DONE

Recorded Acts tokens: 29,999,820


In [39]:
# ============================================================
# STEP 38: LOAD PREPROCESSING STATE
# Restore completed-source information for safe continuation
# ============================================================

def load_preprocessing_state():

    if not os.path.exists(PREPROCESSING_STATE_PATH):

        return {
            "total_train_token_budget": TOTAL_TRAIN_TOKENS,

            "encoder_max_length": ENCODER_MAX_LENGTH,
            "decoder_max_length": DECODER_MAX_LENGTH,

            "prompt_words": PROMPT_WORDS,
            "target_words": TARGET_WORDS,

            "pairs_per_shard": PAIRS_PER_SHARD,

            "sources": {}
        }


    with open(
        PREPROCESSING_STATE_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        state = json.load(file)

    return state


# ------------------------------------------------------------
# Test state loading
# ------------------------------------------------------------

loaded_state = load_preprocessing_state()


print("=" * 70)
print("PREPROCESSING STATE LOADED")
print("=" * 70)

print(
    "Completed sources:",
    list(loaded_state["sources"].keys())
)

print(
    "Acts training tokens:",
    f"{loaded_state['sources']['acts']['train_tokens']:,}"
)

PREPROCESSING STATE LOADED
Completed sources: ['acts']
Acts training tokens: 29,999,820


In [40]:
# ============================================================
# STEP 39: CREATE STATE UPDATE FUNCTION
# Save source completion immediately after preprocessing
# ============================================================

def save_source_to_state(
    source_name,
    source_stats
):

    state = load_preprocessing_state()

    state["sources"][source_name] = source_stats

    with open(
        PREPROCESSING_STATE_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            state,
            file,
            indent=4
        )

    print(
        f"State updated: {source_name} marked as complete."
    )


print("=" * 70)
print("STATE UPDATE FUNCTION")
print("=" * 70)
print("State update function created successfully.")

STATE UPDATE FUNCTION
State update function created successfully.


In [41]:
# ============================================================
# STEP 40: CREATE RESUME-AWARE SOURCE RUNNER
# Skip completed sources and process only unfinished sources
# ============================================================

def run_source_preprocessing(source_name):

    # --------------------------------------------------------
    # Load current preprocessing state
    # --------------------------------------------------------

    state = load_preprocessing_state()


    # --------------------------------------------------------
    # Skip source if already completed
    # --------------------------------------------------------

    if source_name in state["sources"]:

        stats = state["sources"][source_name]

        print("=" * 70)
        print(f"SOURCE ALREADY COMPLETE: {source_name}")
        print("=" * 70)

        print(
            f"Training tokens : "
            f"{stats['train_tokens']:,}"
        )

        print(
            f"Training pairs  : "
            f"{stats['train_pairs']:,}"
        )

        print(
            f"Training shards : "
            f"{stats['train_shards']:,}"
        )

        return stats


    # --------------------------------------------------------
    # Process unfinished source
    # --------------------------------------------------------

    stats = preprocess_source_with_split(
        source_name=source_name,

        source_url=TRAIN_SOURCES[
            source_name
        ],

        train_token_budget=SOURCE_TOKEN_BUDGETS[
            source_name
        ],

        output_dir=FINAL_PROCESSED_DIR,

        pairs_per_shard=PAIRS_PER_SHARD
    )


    # --------------------------------------------------------
    # Record completion immediately
    # --------------------------------------------------------

    save_source_to_state(
        source_name,
        stats
    )


    return stats


print("=" * 70)
print("RESUME-AWARE SOURCE RUNNER")
print("=" * 70)
print("Runner created successfully.")

RESUME-AWARE SOURCE RUNNER
Runner created successfully.


In [42]:
acts_resume_test = run_source_preprocessing("acts")

SOURCE ALREADY COMPLETE: acts
Training tokens : 29,999,820
Training pairs  : 37,717
Training shards : 4


In [43]:
# ============================================================
# STEP 41: PACKAGE TOKENIZER WITH PROCESSED DATA
# Copy BPE vocabulary and merges into the final output folder
# ============================================================

import shutil
import os


FINAL_TOKENIZER_DIR = os.path.join(
    FINAL_PROCESSED_DIR,
    "tokenizer"
)

os.makedirs(
    FINAL_TOKENIZER_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Copy tokenizer files
# ------------------------------------------------------------

for file_name in [
    "vocab.json",
    "merges.txt"
]:

    source_path = os.path.join(
        TOKENIZER_SAVE_DIR,
        file_name
    )

    destination_path = os.path.join(
        FINAL_TOKENIZER_DIR,
        file_name
    )

    shutil.copy2(
        source_path,
        destination_path
    )


# ------------------------------------------------------------
# Verify packaged tokenizer
# ------------------------------------------------------------

print("=" * 70)
print("TOKENIZER PACKAGED")
print("=" * 70)

for file_name in sorted(
    os.listdir(FINAL_TOKENIZER_DIR)
):

    file_path = os.path.join(
        FINAL_TOKENIZER_DIR,
        file_name
    )

    print(
        f"{file_name:<15} "
        f"{os.path.getsize(file_path) / 1024:.2f} KB"
    )

print(
    f"\nSaved at: {FINAL_TOKENIZER_DIR}"
)

TOKENIZER PACKAGED
merges.txt      291.50 KB
vocab.json      500.65 KB

Saved at: /kaggle/working/arch2_final_processed/tokenizer


In [44]:
# ============================================================
# STEP 42: DEFINE REMAINING PRODUCTION RUN
# Process unfinished sources using the resume-aware runner
# ============================================================

REMAINING_SOURCES = [
    "high_courts",
    "supreme_court",
    "tribunals",
]


def run_remaining_preprocessing():

    results = {}

    print("=" * 70)
    print("REMAINING PRODUCTION PREPROCESSING")
    print("=" * 70)

    for source_name in REMAINING_SOURCES:

        print("\n" + "#" * 70)
        print(f"STARTING: {source_name}")
        print("#" * 70)

        stats = run_source_preprocessing(
            source_name
        )

        results[source_name] = stats

        print("\n" + "-" * 70)
        print(f"FINISHED: {source_name}")
        print("-" * 70)

        print(
            f"Training tokens : "
            f"{stats['train_tokens']:,}"
        )

        print(
            f"Training pairs  : "
            f"{stats['train_pairs']:,}"
        )

        print(
            f"Training shards : "
            f"{stats['train_shards']:,}"
        )


    print("\n" + "=" * 70)
    print("ALL REMAINING SOURCES COMPLETE")
    print("=" * 70)

    return results


print("=" * 70)
print("PRODUCTION RUNNER READY")
print("=" * 70)

print("Sources:")
for source_name in REMAINING_SOURCES:
    print(f"  {source_name}")

print("\nFunction created successfully.")

PRODUCTION RUNNER READY
Sources:
  high_courts
  supreme_court
  tribunals

Function created successfully.


In [45]:
# ============================================================
# STEP 43: RUN FINAL PRODUCTION PREPROCESSING
# Process all four sources and save the complete dataset
# ============================================================

PRODUCTION_SOURCES = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]


production_results = {}


print("=" * 70)
print("FINAL PRODUCTION PREPROCESSING")
print("=" * 70)


for source_name in PRODUCTION_SOURCES:

    print("\n" + "#" * 70)
    print(f"STARTING SOURCE: {source_name}")
    print("#" * 70)


    stats = run_source_preprocessing(
        source_name
    )


    production_results[source_name] = stats


    print("\n" + "-" * 70)
    print(f"COMPLETED SOURCE: {source_name}")
    print("-" * 70)

    print(
        f"Training tokens : "
        f"{stats['train_tokens']:,}"
    )

    print(
        f"Training pairs  : "
        f"{stats['train_pairs']:,}"
    )

    print(
        f"Validation pairs: "
        f"{stats['validation_pairs']:,}"
    )

    print(
        f"Training shards : "
        f"{stats['train_shards']:,}"
    )

    print(
        f"Validation shards: "
        f"{stats['validation_shards']:,}"
    )


print("\n" + "=" * 70)
print("ALL PRODUCTION SOURCES COMPLETE")
print("=" * 70)


total_train_tokens = sum(
    stats["train_tokens"]
    for stats in production_results.values()
)

total_train_pairs = sum(
    stats["train_pairs"]
    for stats in production_results.values()
)

total_validation_pairs = sum(
    stats["validation_pairs"]
    for stats in production_results.values()
)


print(
    f"Total training tokens : "
    f"{total_train_tokens:,}"
)

print(
    f"Total training pairs  : "
    f"{total_train_pairs:,}"
)

print(
    f"Total validation pairs: "
    f"{total_validation_pairs:,}"
)

print(
    f"\nFinal dataset directory:\n"
    f"{FINAL_PROCESSED_DIR}"
)

FINAL PRODUCTION PREPROCESSING

######################################################################
STARTING SOURCE: acts
######################################################################
SOURCE ALREADY COMPLETE: acts
Training tokens : 29,999,820
Training pairs  : 37,717
Training shards : 4

----------------------------------------------------------------------
COMPLETED SOURCE: acts
----------------------------------------------------------------------
Training tokens : 29,999,820
Training pairs  : 37,717
Validation pairs: 1,688
Training shards : 4
Validation shards: 1

######################################################################
STARTING SOURCE: high_courts
######################################################################
PROCESSING SOURCE: high_courts
Training token budget: 60,000,000
Train shard 00000 | pairs=10,000 | tokens=7,881,915
Train shard 00001 | pairs=20,000 | tokens=15,754,685
Train shard 00002 | pairs=30,000 | tokens=23,630,190
Train shard 00003 | 